# Tutorial

## Loading packages

In [1]:
from knn_normalization import knn_normalize
import scanpy as sc
import muon as mu

/home/marchena/miniconda3/envs/python_3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading the data

We use our Immune Colon data for this tutorial (SAY HOW TO DOWNLOAD IT).

In [2]:
prot = sc.read_h5ad(
    "/groups/nils/members/javier/Immune_colon/annotation_CITE-seq_CRC/data/CRC_CITE-seq_batch_corrected_adata_v35.h5ad"
)
rna = sc.read_h5ad("/groups/nils/members/javier/Immune_colon/annotation_CITE-seq_CRC/data/rna_v15.h5ad")

In [3]:
prot

AnnData object with n_obs × n_vars = 4075 × 61
    obs: 'has_ir', 'doublet_score', 'predicted_doublet', 'patient', 'date_operation', 'icdo3_topografie', 'icdo3_morphologie', 'tnm_stage', 'tnm_grade', 'previous_therapy', 'diagnosis_free_text', 'origin', 'material', 'snap_frozen', 'ffpe', 'MS_status', 'sample', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'percent_hemo', 'n_counts', 'n_genes', 'S_score', 'G2M_score', 'phase', 'leiden_vanilla', 'leiden_vanilla_highres', 'leiden_harmony', 'leiden_harmony_highres', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size_original', 'clonal_expansion_original', 'epi_score', 'str_score', 'imm_score', 'celltype_1', 'cnv_leiden', 'cnv_score', 'cnv_status', 'raw_protein_counts', 'predicted_cell_type', 'Smooth_cell_type', 'total_counts', 'log_total_counts', 'batch', '_scvi_batch', '_scvi_labels', '_scvi_sample_id', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.7', 'le

In [4]:
rna

AnnData object with n_obs × n_vars = 4075 × 25154
    obs: 'has_ir', 'doublet_score', 'predicted_doublet', 'patient', 'date_operation', 'icdo3_topografie', 'icdo3_morphologie', 'tnm_stage', 'tnm_grade', 'previous_therapy', 'diagnosis_free_text', 'origin', 'material', 'snap_frozen', 'ffpe', 'MS_status', 'sample', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'percent_hemo', 'n_counts', 'n_genes', 'S_score', 'G2M_score', 'phase', 'leiden_vanilla', 'leiden_vanilla_highres', 'leiden_harmony', 'leiden_harmony_highres', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size_original', 'clonal_expansion_original', 'epi_score', 'str_score', 'imm_score', 'celltype_1', 'cnv_leiden', 'cnv_score', 'cnv_status', 'raw_protein_counts', 'predicted_cell_type', 'Smooth_cell_type', 'annotation_protein', 'annotation_protein_v2_l2', 'annotation_protein_v2_l1', 'TCR', 'annotation_combined_v1_l1', 'annotation_combined_v1_l2', 'clone_size', 'clonal_expansion', 'smooth_cell_typ

In [5]:
# This code is only as long as the prot and rna adatas don't have the raw counts in .X
prot.X = prot.layers["raw"]
rna.X = rna.layers["cellranger_counts"]

In [6]:
prot.X

array([[ 17.,   9.,  34., ..., 132.,  27.,  18.],
       [ 91.,   8.,  34., ...,  95.,   7.,  28.],
       [ 18.,   8.,  39., ..., 128.,   9.,   7.],
       ...,
       [ 12.,   2.,  15., ..., 152.,  21.,   4.],
       [ 24.,  20.,  31., ..., 205.,   6.,  13.],
       [ 15.,   2.,  12., ...,  97.,   2.,   6.]], shape=(4075, 61))

MAKE THE ADATAS WAY MORE SIMPLIFIED, WITH ONLY THE DATA, A FEW METADATA COLUMNS AND A FEW EMBEDDINGS AND LAYERS.

Our prot.X consists of raw counts, as is expected by KNN normalization.

We now build a MuData object (LINK TO WHAT A MUDATA OBJECT IS) with the protein and RNA data.

In [7]:
mdata = mu.MuData({"prot": prot, "rna": rna})

/home/marchena/miniconda3/envs/python_3.14/lib/python3.14/site-packages/mudata/_core/mudata.py:988: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  self._update_attr("var", axis=1, join_common=join_common)


In [8]:
mdata

MuData object with n_obs × n_vars = 4075 × 25215
  2 modalities
    prot:	4075 × 61
      obs:	'has_ir', 'doublet_score', 'predicted_doublet', 'patient', 'date_operation', 'icdo3_topografie', 'icdo3_morphologie', 'tnm_stage', 'tnm_grade', 'previous_therapy', 'diagnosis_free_text', 'origin', 'material', 'snap_frozen', 'ffpe', 'MS_status', 'sample', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'percent_hemo', 'n_counts', 'n_genes', 'S_score', 'G2M_score', 'phase', 'leiden_vanilla', 'leiden_vanilla_highres', 'leiden_harmony', 'leiden_harmony_highres', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size_original', 'clonal_expansion_original', 'epi_score', 'str_score', 'imm_score', 'celltype_1', 'cnv_leiden', 'cnv_score', 'cnv_status', 'raw_protein_counts', 'predicted_cell_type', 'Smooth_cell_type', 'total_counts', 'log_total_counts', 'batch', '_scvi_batch', '_scvi_labels', '_scvi_sample_id', 'leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leide

## Running KNN normalization with an MuData object (using neighbors from RNA)

### Run KNN normalization on the MuData object, using calculate_neighbors_from="rna"

In [ ]:
knn_normalize(mdata, calculate_neighbors_from="rna")

Calculating KNN graph from RNA data
Running KNN normalization
Iteration 1
Iteration 2
Change wrt previous iteration: 4.410462075268219
Iteration 3
Change wrt previous iteration: 0.14052335864680954
Iteration 4
Change wrt previous iteration: 0.030036404620623625
Iteration 5
Change wrt previous iteration: 0.01783338928581113
Iteration 6
Change wrt previous iteration: 0.01193127566715435
Iteration 7
Change wrt previous iteration: 0.008675463146150308
Iteration 8
Change wrt previous iteration: 0.006392333536998196
Iteration 9
Change wrt previous iteration: 0.004796440840385996
Iteration 10
Change wrt previous iteration: 0.0036681909851548067
Iteration 11
Change wrt previous iteration: 0.0028589571207164877
Iteration 12
Change wrt previous iteration: 0.002268838445801613
Iteration 13
Change wrt previous iteration: 0.0018308860579885344
Iteration 14
Change wrt previous iteration: 0.0015946711866156038
Iteration 15
Change wrt previous iteration: 0.001446521863407102
Iteration 16
Change wrt pr

In [10]:
mdata["prot"].X

array([[3.03909437, 2.58710925, 3.61161357, ..., 4.86803285, 3.41378782,
        3.08354614],
       [4.61042843, 2.61102959, 3.70964188, ..., 4.65125042, 2.53098689,
        3.5425878 ],
       [3.37909102, 2.80854616, 4.02778644, ..., 5.13394593, 2.88265414,
        2.72850346],
       ...,
       [3.29102478, 2.40372158, 3.45354371, ..., 5.51405724, 3.71590797,
        2.65503601],
       [3.58209207, 3.43367206, 3.79831517, ..., 5.56190377, 2.61269151,
        3.10516799],
       [3.75867932, 2.70885719, 3.59616039, ..., 5.38791986, 2.70885719,
        3.16084232]], shape=(4075, 61))

The ``knn_normalize`` function expects raw counts in protein and RNA, since by default it normalizes and log-transforms the RNA data before calculating the KNN graph from the RNA data, then KNN-normalizes the protein data and then log-transforms the KNN-normalized protein data. If you want to use KNN normalized with already-processed data, see the section "Running KNN normalization on a pre-calculated KNN graph".

If you don't want to print progress messages, pass ``verbose=False``. 

## Running KNN normalization with an AnnData object (using neighbors from protein)

In [11]:
prot = sc.read_h5ad(
    "/groups/nils/members/javier/Immune_colon/annotation_CITE-seq_CRC/data/CRC_CITE-seq_batch_corrected_adata_v35.h5ad"
)

In [12]:
# This code is only as long as the prot and rna adatas don't have the raw counts in .X
prot.X = prot.layers["raw"]
rna.X = rna.layers["cellranger_counts"]

In [ ]:
knn_normalize(prot, calculate_neighbors_from="prot")

Calculating KNN graph from protein data
Running KNN normalization
Iteration 1
Iteration 2
Change wrt previous iteration: 3.328236350732083
Iteration 3
Change wrt previous iteration: 0.19034927698508963
Iteration 4
Change wrt previous iteration: 0.048138129300577925
Iteration 5
Change wrt previous iteration: 0.019618969787803198
Iteration 6
Change wrt previous iteration: 0.01062573261936861
Iteration 7
Change wrt previous iteration: 0.008457378174948182
Iteration 8
Change wrt previous iteration: 0.00664046469210855
Iteration 9
Change wrt previous iteration: 0.005158215097124552
Iteration 10
Change wrt previous iteration: 0.003996297241218194
Iteration 11
Change wrt previous iteration: 0.0030967938365917824
Iteration 12
Change wrt previous iteration: 0.002401091528317245
Iteration 13
Change wrt previous iteration: 0.0018773886555232266
Iteration 14
Change wrt previous iteration: 0.001471450907231997
Iteration 15
Change wrt previous iteration: 0.0011545343369949723
Iteration 16
Change wrt

In [14]:
prot.X

array([[3.15055845, 2.69857333, 3.72307765, ..., 4.97949693, 3.5252519 ,
        3.19501022],
       [4.63205   , 2.63265117, 3.73126346, ..., 4.672872  , 2.55260846,
        3.56420937],
       [3.41843857, 2.84789371, 4.06713398, ..., 5.17329348, 2.92200168,
        2.767851  ],
       ...,
       [3.36126193, 2.47395873, 3.52378086, ..., 5.58429439, 3.78614512,
        2.72527316],
       [3.64110846, 3.49268845, 3.85733157, ..., 5.62092016, 2.6717079 ,
        3.16418439],
       [3.88471098, 2.83488885, 3.72219205, ..., 5.51395151, 2.83488885,
        3.28687397]], shape=(4075, 61))

## Running KNN normalization on a user-computed KNN graph

Let's say you want to run KNN normalization but use your own KNN graph built with your own favorite method and data. This is only recommended if you think your KNN graph is more precise than our default KNN graph building procedure.

As an example, we normalize, log-transform, select 2000 HVGs and run PCA on the RNA data and use 20 PCs to calculate RNA neighbors.

In [15]:
prot = sc.read_h5ad(
    "/groups/nils/members/javier/Immune_colon/annotation_CITE-seq_CRC/data/CRC_CITE-seq_batch_corrected_adata_v35.h5ad"
)
rna = sc.read_h5ad("/groups/nils/members/javier/Immune_colon/annotation_CITE-seq_CRC/data/rna_v15.h5ad")
mdata = mu.MuData({"prot": prot, "rna": rna})

/home/marchena/miniconda3/envs/python_3.14/lib/python3.14/site-packages/mudata/_core/mudata.py:988: UserWarning: var_names are not unique. To make them unique, call `.var_names_make_unique`.
  self._update_attr("var", axis=1, join_common=join_common)


In [16]:
# This code is only as long as the prot and rna adatas don't have the raw counts in .X
prot.X = prot.layers["raw"]
rna.X = rna.layers["cellranger_counts"]
del rna.uns["log1p"]

In [17]:
sc.pp.normalize_total(mdata["rna"])
sc.pp.log1p(mdata["rna"])
sc.pp.highly_variable_genes(mdata["rna"], n_top_genes=2000, batch_key="sample")
sc.tl.pca(mdata["rna"], n_comps=20)
sc.pp.neighbors(mdata["rna"])

The knn_normalize() function uses neighbors from adata.obsp["connectivities"], therefore we now transfer the KNN graph from mdata["rna"].obsp["connectivities"] to mdata["prot"].obsp["connectivities"].

In [18]:
mdata["prot"].obsp["connectivities"] = mdata["rna"].obsp["connectivities"]

In [ ]:
knn_normalize(mdata, calculate_neighbors_from="use_existing_neighbors")

Running KNN normalization
Iteration 1
Iteration 2
Change wrt previous iteration: 4.243506674807613
Iteration 3
Change wrt previous iteration: 0.4805265533853337
Iteration 4
Change wrt previous iteration: 0.1221497705484641
Iteration 5
Change wrt previous iteration: 0.039433062434657384
Iteration 6
Change wrt previous iteration: 0.02096692142976786
Iteration 7
Change wrt previous iteration: 0.009064230159924636
Iteration 8
Change wrt previous iteration: 0.006513672018952477
Iteration 9
Change wrt previous iteration: 0.004710409148070083
Iteration 10
Change wrt previous iteration: 0.003587321262331411
Iteration 11
Change wrt previous iteration: 0.002715482798557378
Iteration 12
Change wrt previous iteration: 0.002083390433549548
Iteration 13
Change wrt previous iteration: 0.0016782217155024703
Iteration 14
Change wrt previous iteration: 0.0013899631463675233
Iteration 15
Change wrt previous iteration: 0.0011518399547216784
Iteration 16
Change wrt previous iteration: 0.0009553929948817874

In [21]:
mdata["prot"].X

array([[2.84774353, 2.39575841, 3.42026273, ..., 4.67668201, 3.22243698,
        2.8921953 ],
       [4.57042811, 2.57102928, 3.66964157, ..., 4.61125011, 2.49098657,
        3.50258748],
       [3.1957257 , 2.62518084, 3.84442112, ..., 4.95058062, 2.69928882,
        2.54513814],
       ...,
       [3.45974987, 2.57244667, 3.6222688 , ..., 5.68278233, 3.88463306,
        2.8237611 ],
       [3.79237685, 3.64395685, 4.00859996, ..., 5.77218855, 2.82297629,
        3.31545278],
       [3.93539687, 2.88557475, 3.77287794, ..., 5.56463741, 2.88557475,
        3.33755987]], shape=(4075, 61))

## Using KNN normalization after dsb normalization

Let's say you first want to dsb-normalize (CITE PAPER) your protein data and then use KNN normalization. This is possible. Other ways to process the protein data before KNN normalization are also technically possible, but make sure your pipeline is appropriate.

In [ ]:
import lamindb as ln

af = ln.Artifact.connect("theislab/sc-best-practices").get(
    key="surface-protein/cite_normalization.h5mu", is_latest=True
)
mdata = af.load()
mdata

! run input wasn't tracked, call `ln.track()` and re-run


MuData object with n_obs × n_vars = 106515 × 36741
  var:	'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
  2 modalities
    rna:	106515 × 36601
      obs:	'donor', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'outliers'
      var:	'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
      layers:	None
    prot:	106515 × 140
      obs:	'donor', 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'n_genes', 'n_counts', 'outliers'
      var:	'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
      layer

In [59]:
mdata["prot"].X

array([[ 0.41156265,  5.1750746 , 12.487137  , ..., 19.187222  ,
         8.098491  , 21.78619   ],
       [69.36184   ,  3.0661297 ,  7.2966785 , ..., 18.108246  ,
         5.6975536 , 20.831774  ],
       [ 4.6376553 ,  3.6098168 , 12.17653   , ...,  7.05072   ,
         7.774398  , 10.512558  ],
       ...,
       [57.55452   ,  1.47561   , 15.048151  , ..., 30.503693  ,
         4.300117  , 11.297047  ],
       [-3.858018  ,  1.4442642 ,  5.8381157 , ...,  4.4191113 ,
         3.26615   ,  2.8864465 ],
       [-4.521755  ,  4.925741  ,  5.2978897 , ..., 12.427093  ,
         2.3641114 ,  4.0995736 ]], shape=(106515, 140), dtype=float32)

In [65]:
mdata["rna"].X[15:30, 50:60].toarray()

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 4., 0., 0., 0., 0., 0., 9.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 1.],
       [0., 1., 0., 0., 0., 2., 0., 0., 0., 4.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 2., 0., 2.],
       [0., 1., 0., 1., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 1.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 2.],
       [0., 0., 1., 1., 0., 1., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 1.]], dtype=float32)

We do not log-transform or use pseudocounts since dsb normalization already used pseudocounts and log-transformed the data. We run KNN normalization only with 5 iterations since the dataset is relatively big and KNN takes a bit of time.

Ignore the warnings that say protein data is not integer-valued and contains negative values, since dsb normalization created negative float values, but that's fine.

In [ ]:
knn_normalize(mdata, calculate_neighbors_from="rna", max_iterations=5, log_transform=False, pseudocount=0)

/groups/nils/members/javier/KNN_normalization/src/knn_normalization/tl/tools.py:85: UserWarning: The protein data contains negative values; raw (non-negative) counts are expected.
  _validate_inputs(
/tmp/ipykernel_274297/1749884609.py:1: UserWarning: The protein data is not integer-valued. KNN normalization expects raw counts.
  knn.knn_normalize(mdata, calculate_neighbors_from="rna")


Calculating KNN graph from RNA data
Running KNN normalization
Iteration 1
Iteration 2
Change wrt previous iteration: 1.86830377519005
Iteration 3
Change wrt previous iteration: 0.14769218436038212
Iteration 4
Change wrt previous iteration: 0.020572023464034528
Iteration 5
Change wrt previous iteration: 0.009542967969133342
Iteration 6
Change wrt previous iteration: 0.006781421215018568
Iteration 7
Change wrt previous iteration: 0.005005326030937729
Iteration 8
Change wrt previous iteration: 0.003967761058296881
Iteration 9
Change wrt previous iteration: 0.0031895892831879102
Iteration 10
Change wrt previous iteration: 0.0025825081293461283
Iteration 11
Change wrt previous iteration: 0.002078860457736309
Iteration 12
Change wrt previous iteration: 0.0016711055583813472
Iteration 13
Change wrt previous iteration: 0.0013465523657079093
Iteration 14
Change wrt previous iteration: 0.0010827680043981047
Iteration 15
Change wrt previous iteration: 0.0008696881304440796
Iteration 16
Change wrt

/groups/nils/members/javier/KNN_normalization/src/knn_normalization/tl/tools.py:182: RuntimeWarning: invalid value encountered in log
  np.log(x, out=x)


In [6]:
mdata["prot"].X

array([[ 1.6360482 ,  2.26745136,  2.80897588, ...,  3.13333476,
         2.52000732,  3.23539676],
       [ 4.22023403,  1.99896491,  2.42062032, ...,  3.05148065,
         2.28130621,  3.16289641],
       [ 2.10358957,  1.99081476,  2.68145563, ...,  2.32703615,
         2.38535472,  2.57956161],
       ...,
       [ 3.86354405,  1.59554837,  2.72564249, ...,  3.29714227,
         1.95753254,  2.51848946],
       [ 0.26383192,  1.99425696,  2.51413565, ...,  2.37380724,
         2.24323536,  2.19621215],
       [-0.64372626,  2.38903783,  2.42584532, ...,  2.95193237,
         2.09052472,  2.30213388]], shape=(106515, 140))